# Stage 2 — Data Cleaning, Decision Freeze, and Preprocessing Specification

This notebook implements the decisions made after exploratory data analysis. It does **not** train a prediction model.

## Outputs

- machine-readable decision file;
- source-file checksum;
- auditable correction log;
- cleaned master dataset;
- three target-specific primary datasets;
- three expanded sensitivity datasets;
- complete target-specific predictor manifests;
- reusable, unfitted scikit-learn preprocessing factory;
- automated validation tables;
- a clean, self-contained HTML execution report.

## Safety principles

1. The original CSV is never overwritten.
2. Every value correction is tied to `VisitID` and validated against the expected original value.
3. `VisitID` is retained for traceability but never listed as a predictor.
4. The other outcomes and `OnlyLargeJoints` never enter a predictor matrix.
5. Imputation, encoding, scaling, and model fitting are deferred to future training folds.


In [ ]:
# 1. Imports and environment
from __future__ import annotations

import hashlib
import json
import os
import platform
import subprocess
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import yaml
from IPython.display import display, Markdown

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

pd.set_option('display.max_columns', 140)
pd.set_option('display.max_rows', 140)
pd.set_option('display.width', 190)
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')

print('Python:', sys.version.split()[0])
print('Platform:', platform.platform())
print('pandas:', pd.__version__)
import sklearn
print('scikit-learn:', sklearn.__version__)


In [ ]:
# 2. Path configuration
CSV_NAME = 'Atypical_presentations_RA_predictors_valid_c_exclusions_final_population.csv'
NOTEBOOK_NAME = '02_data_cleaning_and_preprocessing.ipynb'
ENV_PROJECT = Path(os.environ['RA_PROJECT_DIR']).expanduser() if os.environ.get('RA_PROJECT_DIR') else Path.cwd()

candidate_paths = [
    ENV_PROJECT / CSV_NAME,
    Path.cwd() / CSV_NAME,
    Path.cwd() / 'uploads' / CSV_NAME,
    Path(os.environ['RA_DATA_PATH']).expanduser() if os.environ.get('RA_DATA_PATH') else Path.cwd() / 'uploads' / CSV_NAME,
]
SOURCE_PATH = next((p for p in candidate_paths if p.exists()), None)
if SOURCE_PATH is None:
    checked = '\n'.join(f'  - {p}' for p in candidate_paths)
    raise FileNotFoundError(f'Could not find {CSV_NAME}. Checked:\n{checked}')

PROJECT_DIR = ENV_PROJECT if ENV_PROJECT.exists() else Path.cwd() / 'stage2_outputs'
CONFIG_DIR = PROJECT_DIR / 'config'
CLEAN_DIR = PROJECT_DIR / 'cleaned_data'
DERIVED_DIR = PROJECT_DIR / 'derived_data'
PREPROCESS_DIR = PROJECT_DIR / 'preprocessing'
REPORT_DIR = PROJECT_DIR / 'preprocessing_outputs' / 'report'
TABLE_DIR = PROJECT_DIR / 'preprocessing_outputs' / 'tables'
for directory in [CONFIG_DIR, CLEAN_DIR, DERIVED_DIR, PREPROCESS_DIR, REPORT_DIR, TABLE_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print('Source:', SOURCE_PATH.resolve())
print('Project:', PROJECT_DIR.resolve())


## Freeze the scientific decisions

In [ ]:
# 3. Machine-readable decision specification
CORRECTIONS_PATH = os.environ.get('RA_CORRECTIONS_PATH')
if not CORRECTIONS_PATH:
    raise RuntimeError(
        'RA_CORRECTIONS_PATH is required: set it to the owner-supplied, '
        'authorized local correction specification. See .env.example.'
    )
corrections_file = Path(CORRECTIONS_PATH).expanduser()
if not corrections_file.is_file():
    raise FileNotFoundError(f'Authorized correction specification not found: {corrections_file}')

def _load_authorized_corrections():
    records = json.loads(corrections_file.read_text(encoding='utf-8'))
    if not isinstance(records, list) or not records:
        raise ValueError('Correction specification must be a non-empty JSON list.')
    required = {'VisitID', 'variable', 'expected_original', 'clean_value', 'reason'}
    for record in records:
        missing = required - set(record)
        if missing:
            raise ValueError(f'Correction record missing keys: {sorted(missing)}')
    return records

DECISIONS = {
    'version': '1.0',
    'stage': 'data_cleaning_and_preprocessing',
    'model_family_status': 'pending_investigator_decision',
    'source_file': CSV_NAME,
    'identifier': 'VisitID',
    'retained_outcomes': [
        'NegRFandACCP',
        'NoHandJointInvolvement',
        'AsymmetricArthritis',
    ],
    'excluded_outcomes': ['OnlyLargeJoints'],
    'estimand': 'leakage-controlled contemporaneous classification at the observed visit',
    # PRIVACY: individual-level correction records (identifier plus original and
    # cleaned values) are withheld from this public code repository. The owner
    # supplies them at an authorized local path; see config/corrections_template.json
    # and .env.example. The validation logic below is unchanged.
    'fixed_value_corrections': _load_authorized_corrections(),
    'engineered_predictors': {
        'MeanGripStrength': {
            'source_columns': ['PGAGripStrengthRight', 'PGAGripStrengthLeft'],
            'rule': 'row mean of valid hands; one valid hand is used; missing if both missing',
        }
    },
    'global_primary_exclusions': {
        'identifier_DO_NOT_MODEL': ['VisitID'],
        'constant_or_no_variation': ['NowPregnant', 'Cancer', 'NotCurrentlyClassifiableasRA2010'],
        'reproductive_female_only': [
            'Menopause', 'PregnancyHistory', 'NoOfPregnancy', 'PregnancyAgeOf1thPregnancy',
            'FetalLoss', 'ContraceptionMethodOCP', 'ContraceptionMethodIUD', 'PrematureInfants',
        ],
        'high_missingness_sensitivity_only': ['BloodGroup', 'BloodRH', 'DLPObj'],
        'extreme_rarity_excluded': ['InitialSiteExtraArticular', 'CNSCVA'],
        'overlapping_alternative_sensitivity': ['IllicitDrug'],
        'pattern_variables_sensitivity_only': ['PolyarticularType', 'JointInvolvementPattern'],
        'grip_components_alternative_sensitivity': ['PGAGripStrengthRight', 'PGAGripStrengthLeft'],
    },
    'primary_retained_overlapping_exposure': 'Opium',
    'high_missingness_exception_primary': 'TyG',
    'target_specific_primary_exclusions': {
        'NegRFandACCP': ['AntiCCPPosNeg', 'SerumRheumatoidFactorPositive'],
        'NoHandJointInvolvement': [
            'DIPInvolvementBinary', 'HandPIPInvolvementBinary',
            'MCPInvolvementB', 'WristInvolvementBinary',
        ],
        'AsymmetricArthritis': ['JointInvolvementPattern'],
    },
    'outcome_label_sensitivity': {
        'NegRFandACCP': 'primary uses recorded labels; sensitivity excludes complete-serology discordant records (count withheld)',
        'NoHandJointInvolvement': 'primary uses recorded labels; do not reconstruct from hand markers',
        'AsymmetricArthritis': 'primary uses recorded labels; joint pattern is sensitivity-only',
    },
    'rarity_rule': {
        'binary': 'exclude from primary when either observed class has fewer than 10 subjects',
        'categorical': 'combine a rare level only when clinically defensible; otherwise sensitivity-only',
    },
    'missing_data_primary': {
        'numeric': 'training-fold median imputation and scaling',
        'binary': 'training-fold most-frequent imputation plus missingness indicators',
        'nominal': 'explicit Unknown level then one-hot encoding with unseen-category protection',
        'TyG': 'retain in primary; set isolated value missing; include missingness indication through numeric imputer',
    },
    'missing_data_sensitivity': 'iterative/multivariable imputation inside resampling',
    'class_imbalance_primary': 'unweighted loss with stratified resampling; no SMOTE or undersampling',
    'class_imbalance_sensitivity': 'balanced class weighting',
    'validation': {
        'outer': 'stratified 5-fold repeated 10 times',
        'inner': 'stratified 5-fold for tuning',
        'performance_predictions': 'out-of-fold only; average 10 predictions per subject',
        'uncertainty': 'subject-level bootstrap confidence intervals',
    },
}

DECISION_PATH = CONFIG_DIR / 'analysis_decisions.yaml'
with open(DECISION_PATH, 'w', encoding='utf-8') as f:
    yaml.safe_dump(DECISIONS, f, sort_keys=False, allow_unicode=True)
print('Saved:', DECISION_PATH)
display(pd.DataFrame({'retained_outcome': DECISIONS['retained_outcomes']}))


## Load and fingerprint the unchanged source file

In [ ]:
# 4. Cryptographic checksum and source validation
def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

source_hash_before = sha256_file(SOURCE_PATH)
raw = pd.read_csv(SOURCE_PATH)

required = {'VisitID', 'OnlyLargeJoints', *DECISIONS['retained_outcomes']}
missing_required = sorted(required - set(raw.columns))
if missing_required:
    raise ValueError(f'Required columns missing: {missing_required}')
if raw.columns.duplicated().any():
    raise ValueError('Duplicated column names detected.')
if raw['VisitID'].isna().any() or not raw['VisitID'].is_unique:
    raise ValueError('VisitID must be complete and unique.')
if len(raw) != 820 or raw.shape[1] != 95:
    raise ValueError(f'Unexpected source shape {raw.shape}; expected (820, 95).')

source_fingerprint = pd.DataFrame([{
    'source_path': str(SOURCE_PATH.resolve()),
    'sha256': source_hash_before,
    'rows': len(raw),
    'columns': raw.shape[1],
    'unique_VisitID': raw.VisitID.nunique(),
    'exact_duplicate_rows': int(raw.duplicated().sum()),
}])
source_fingerprint.to_csv(CLEAN_DIR / 'source_fingerprint.csv', index=False, encoding='utf-8-sig')
display(source_fingerprint)


## Apply only the agreed corrections

In [ ]:
# 5. Auditable correction function
clean = raw.copy(deep=True)
correction_rows = []

for correction in DECISIONS['fixed_value_corrections']:
    visit_id = correction['VisitID']
    variable = correction['variable']
    expected = correction['expected_original']
    clean_value = correction['clean_value']
    mask = clean['VisitID'].eq(visit_id)
    if mask.sum() != 1:
        raise ValueError(f'Expected exactly one row for VisitID {visit_id}; found {mask.sum()}.')
    observed = clean.loc[mask, variable].iloc[0]
    if not np.isclose(observed, expected, equal_nan=True, rtol=0, atol=1e-12):
        raise ValueError(
            f'Correction precondition failed for VisitID {visit_id}, {variable}: '
            f'expected {expected}, found {observed}. Source may have changed.'
        )
    new_value = np.nan if clean_value is None else float(clean_value)
    clean.loc[mask, variable] = new_value
    correction_rows.append({
        'VisitID': visit_id, 'variable': variable, 'original_value': observed,
        'clean_value': new_value, 'reason': correction['reason'],
        'decision_version': DECISIONS['version'],
    })

correction_log = pd.DataFrame(correction_rows)
correction_log.to_csv(CLEAN_DIR / 'correction_log.csv', index=False, encoding='utf-8-sig')
display(correction_log)

# Derived bilateral grip value is created only after corrections.
clean['MeanGripStrength'] = clean[['PGAGripStrengthRight', 'PGAGripStrengthLeft']].mean(axis=1, skipna=True)
clean.loc[clean[['PGAGripStrengthRight', 'PGAGripStrengthLeft']].isna().all(axis=1), 'MeanGripStrength'] = np.nan

# Verify that outcomes were not altered.
for outcome in DECISIONS['retained_outcomes'] + DECISIONS['excluded_outcomes']:
    if not raw[outcome].equals(clean[outcome]):
        raise AssertionError(f'Outcome was unexpectedly changed: {outcome}')

CLEAN_MASTER_PATH = CLEAN_DIR / 'cleaned_master_data.csv'
clean.to_csv(CLEAN_MASTER_PATH, index=False, encoding='utf-8-sig')
print('Saved:', CLEAN_MASTER_PATH)
print('Cleaned shape:', clean.shape)


In [ ]:
# 6. Confirm exact correction results
correction_checks = []
for c in DECISIONS['fixed_value_corrections']:
    value = clean.loc[clean.VisitID.eq(c['VisitID']), c['variable']].iloc[0]
    expected_clean = np.nan if c['clean_value'] is None else c['clean_value']
    passed = pd.isna(value) if pd.isna(expected_clean) else np.isclose(value, expected_clean)
    correction_checks.append({
        'VisitID': c['VisitID'], 'variable': c['variable'],
        'expected_clean_value': expected_clean, 'observed_clean_value': value,
        'status': 'PASS' if passed else 'FAIL',
    })
correction_checks = pd.DataFrame(correction_checks)
display(correction_checks)
if not correction_checks.status.eq('PASS').all():
    raise AssertionError('One or more corrections failed.')

# Raw file must remain byte-for-byte unchanged.
source_hash_after = sha256_file(SOURCE_PATH)
if source_hash_after != source_hash_before:
    raise AssertionError('The source CSV changed during cleaning.')
print('Source checksum unchanged:', source_hash_after)


## Define semantic types and outcome-specific roles

In [ ]:
# 7. Semantic typing after cleaning
IDENTIFIER = DECISIONS['identifier']
RETAINED_OUTCOMES = DECISIONS['retained_outcomes']
ALL_OUTCOMES = RETAINED_OUTCOMES + DECISIONS['excluded_outcomes']


def semantic_type(series: pd.Series) -> str:
    observed = series.dropna()
    unique = set(observed.unique().tolist())
    if pd.api.types.is_object_dtype(series) or isinstance(series.dtype, pd.CategoricalDtype):
        return 'categorical'
    if unique and unique.issubset({0, 1, 0.0, 1.0}):
        return 'constant_binary' if len(unique) == 1 else 'binary'
    if len(unique) == 0:
        return 'all_missing'
    return 'numeric'

semantic_map = {col: semantic_type(clean[col]) for col in clean.columns}
semantic_inventory = pd.DataFrame([
    {
        'column': col, 'semantic_type': semantic_map[col],
        'missing_n': int(clean[col].isna().sum()),
        'missing_pct': 100 * clean[col].isna().mean(),
        'unique_nonmissing_n': int(clean[col].nunique(dropna=True)),
    }
    for col in clean.columns
])
semantic_inventory.to_csv(TABLE_DIR / 'semantic_inventory_after_cleaning.csv', index=False, encoding='utf-8-sig')
display(semantic_inventory)


In [ ]:
# 8. Build a deterministic role assignment for every target
G = DECISIONS['global_primary_exclusions']
constant_excluded = set(G['constant_or_no_variation'])
reproductive_excluded = set(G['reproductive_female_only'])
high_missing_sensitivity = set(G['high_missingness_sensitivity_only'])
rare_excluded = set(G['extreme_rarity_excluded'])
overlap_alternative = set(G['overlapping_alternative_sensitivity'])
pattern_sensitivity = set(G['pattern_variables_sensitivity_only'])
grip_components = set(G['grip_components_alternative_sensitivity'])


def assign_role(column: str, target: str) -> tuple[str, str]:
    other_outcomes = set(ALL_OUTCOMES) - {target}
    target_direct = set(DECISIONS['target_specific_primary_exclusions'][target])
    if column == IDENTIFIER:
        return 'identifier_DO_NOT_MODEL', 'Retained only for tracing rows and predictions.'
    if column == target:
        return 'outcome', 'Binary target for this analysis.'
    if column in other_outcomes:
        if column == 'OnlyLargeJoints':
            return 'excluded_project_outcome', 'Outcome removed from the project; never a predictor.'
        return 'other_outcome_excluded', 'Other atypical outcome; never a predictor.'
    if column in constant_excluded:
        return 'constant_excluded', 'No observed variation.'
    if column in reproductive_excluded:
        return 'reproductive_female_only', 'Excluded from primary all-patient models.'
    if column in rare_excluded:
        return 'rare_excluded', 'Fewer than 10 observations in the minority category.'
    if column in target_direct:
        return 'direct_target_component_sensitivity', 'Possible direct component or near-definitional target proxy.'
    if column in high_missing_sensitivity:
        return 'high_missingness_sensitivity', 'More than 30% missing; TyG is the only prespecified exception.'
    if column in pattern_sensitivity:
        return 'pattern_variable_sensitivity', 'Sparse levels and/or outcome-proximal clinical pattern.'
    if column in overlap_alternative:
        return 'alternative_coding_sensitivity', 'Overlaps strongly with the retained Opium field.'
    if column in grip_components:
        return 'alternative_grip_sensitivity', 'Replaced by MeanGripStrength in primary models.'
    return 'primary_predictor', 'Eligible after prespecified outcome-independent and target-specific rules.'

role_tables = {}
for target in RETAINED_OUTCOMES:
    rows = []
    for col in clean.columns:
        role, reason = assign_role(col, target)
        rows.append({
            'column': col, 'role': role, 'reason': reason,
            'semantic_type': semantic_map[col],
            'missing_n': int(clean[col].isna().sum()),
            'missing_pct': 100 * clean[col].isna().mean(),
        })
    manifest = pd.DataFrame(rows)
    role_tables[target] = manifest
    display(Markdown(f'### Role counts: `{target}`'))
    display(manifest.role.value_counts().rename_axis('role').reset_index(name='n_columns'))


## Generate the three primary and expanded-sensitivity datasets

In [ ]:
# 9. Export target-specific datasets and complete manifests
# Expanded sensitivity includes the primary set plus direct target components,
# high-missingness fields, and pattern fields. Alternative codings (IllicitDrug and
# separate hands) remain in the cleaned master and manifest but are not included
# simultaneously with their primary representation.

expanded_roles = {
    'primary_predictor',
    'direct_target_component_sensitivity',
    'high_missingness_sensitivity',
    'pattern_variable_sensitivity',
}
registry_rows = []

for target in RETAINED_OUTCOMES:
    manifest = role_tables[target]
    primary_predictors = manifest.loc[manifest.role.eq('primary_predictor'), 'column'].tolist()
    expanded_predictors = manifest.loc[manifest.role.isin(expanded_roles), 'column'].tolist()
    analysis_mask = clean[target].notna()

    primary = clean.loc[analysis_mask, [IDENTIFIER, target] + primary_predictors].copy()
    expanded = clean.loc[analysis_mask, [IDENTIFIER, target] + expanded_predictors].copy()

    # Core safety assertions.
    assert primary[IDENTIFIER].is_unique
    assert primary[target].notna().all()
    assert not (set(ALL_OUTCOMES) - {target}) & set(primary.columns)
    assert not set(G['constant_or_no_variation']) & set(primary_predictors)
    assert not set(G['reproductive_female_only']) & set(primary_predictors)
    assert not set(G['high_missingness_sensitivity_only']) & set(primary_predictors)
    assert not set(G['extreme_rarity_excluded']) & set(primary_predictors)
    assert not set(G['overlapping_alternative_sensitivity']) & set(primary_predictors)
    assert not set(G['pattern_variables_sensitivity_only']) & set(primary_predictors)
    assert not set(G['grip_components_alternative_sensitivity']) & set(primary_predictors)
    assert not set(DECISIONS['target_specific_primary_exclusions'][target]) & set(primary_predictors)
    assert 'MeanGripStrength' in primary_predictors
    assert 'TyG' in primary_predictors
    assert 'Opium' in primary_predictors

    primary_path = DERIVED_DIR / f'{target}__primary_eligible.csv'
    expanded_path = DERIVED_DIR / f'{target}__sensitivity_expanded.csv'
    manifest_path = DERIVED_DIR / f'{target}__predictor_manifest.csv'
    primary.to_csv(primary_path, index=False, encoding='utf-8-sig')
    expanded.to_csv(expanded_path, index=False, encoding='utf-8-sig')
    manifest.to_csv(manifest_path, index=False, encoding='utf-8-sig')

    registry_rows.append({
        'outcome': target,
        'analysis_rows': int(analysis_mask.sum()),
        'positive_n': int((clean.loc[analysis_mask, target] == 1).sum()),
        'negative_n': int((clean.loc[analysis_mask, target] == 0).sum()),
        'primary_predictors_n': len(primary_predictors),
        'expanded_predictors_n': len(expanded_predictors),
        'primary_dataset': str(primary_path),
        'expanded_dataset': str(expanded_path),
        'manifest': str(manifest_path),
    })

registry = pd.DataFrame(registry_rows)
registry.to_csv(DERIVED_DIR / 'dataset_registry.csv', index=False, encoding='utf-8-sig')
display(registry)


In [ ]:
# 10. Save alternative sensitivity coding instructions
alternative_sensitivities = pd.DataFrame([
    {
        'sensitivity_id': 'S_ALT_DRUG',
        'change_from_primary': 'Replace Opium with IllicitDrug; never include both simultaneously.',
        'purpose': 'Test dependence on narrow versus broad drug-exposure coding.',
    },
    {
        'sensitivity_id': 'S_ALT_GRIP',
        'change_from_primary': 'Replace MeanGripStrength with corrected right and left grip fields.',
        'purpose': 'Test whether bilateral aggregation changes conclusions.',
    },
    {
        'sensitivity_id': 'S_NO_TYG',
        'change_from_primary': 'Remove TyG and its missingness contribution.',
        'purpose': 'Test dependence on a predictor with 35% missingness.',
    },
    {
        'sensitivity_id': 'S_CLASS_WEIGHT',
        'change_from_primary': 'Use balanced class weighting without changing validation prevalence.',
        'purpose': 'Test sensitivity/specificity trade-off under weighted loss.',
    },
    {
        'sensitivity_id': 'S_ITERATIVE_IMPUTE',
        'change_from_primary': 'Replace simple fold-contained imputation with iterative multivariable imputation.',
        'purpose': 'Test dependence on the missing-data model.',
    },
    {
        'sensitivity_id': 'S_SERO_DISCORDANT',
        'change_from_primary': 'For NegRFandACCP only, exclude 11 complete-serology discordant records.',
        'purpose': 'Test sensitivity to possible outcome-label disagreement.',
    },
])
alternative_sensitivities.to_csv(CONFIG_DIR / 'sensitivity_analysis_plan.csv', index=False, encoding='utf-8-sig')
display(alternative_sensitivities)


## Define reusable preprocessing without fitting it to the full dataset

In [ ]:
# 11. Preprocessing factory used later inside every training fold
def get_predictor_columns(manifest: pd.DataFrame, scope: str = 'primary') -> dict[str, list[str]]:
    if scope == 'primary':
        eligible_roles = {'primary_predictor'}
    elif scope == 'expanded':
        eligible_roles = expanded_roles
    else:
        raise ValueError("scope must be 'primary' or 'expanded'")

    selected = manifest[manifest.role.isin(eligible_roles)].copy()
    return {
        'numeric': selected.loc[selected.semantic_type.eq('numeric'), 'column'].tolist(),
        'binary': selected.loc[selected.semantic_type.eq('binary'), 'column'].tolist(),
        'categorical': selected.loc[selected.semantic_type.eq('categorical'), 'column'].tolist(),
    }


def make_preprocessor(column_groups: dict[str, list[str]]) -> ColumnTransformer:
    # This object is intentionally returned UNFITTED. Future modeling code must clone
    # and fit it using training folds only.
    numeric_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='median', add_indicator=True, keep_empty_features=True)),
        ('scaler', StandardScaler()),
    ])
    binary_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent', add_indicator=True, keep_empty_features=True)),
    ])
    categorical_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='constant', fill_value='Unknown', keep_empty_features=True)),
        ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=True)),
    ])
    return ColumnTransformer(
        transformers=[
            ('numeric', numeric_pipeline, column_groups['numeric']),
            ('binary', binary_pipeline, column_groups['binary']),
            ('categorical', categorical_pipeline, column_groups['categorical']),
        ],
        remainder='drop',
        verbose_feature_names_out=True,
    )

preprocessing_specs = {}
for target in RETAINED_OUTCOMES:
    groups = get_predictor_columns(role_tables[target], scope='primary')
    preprocessing_specs[target] = groups
    preprocessor = make_preprocessor(groups)
    # Confirm object exists but is not fitted.
    assert not hasattr(preprocessor, 'transformers_')
    print(target, {k: len(v) for k, v in groups.items()}, '| fitted:', hasattr(preprocessor, 'transformers_'))

with open(PREPROCESS_DIR / 'preprocessing_columns.json', 'w', encoding='utf-8') as f:
    json.dump(preprocessing_specs, f, indent=2, ensure_ascii=False)


In [ ]:
# 12. Write the reusable preprocessing factory as a standalone Python module
factory_source = r"""from __future__ import annotations

import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

PRIMARY_ROLES = {'primary_predictor'}
EXPANDED_ROLES = {
    'primary_predictor',
    'direct_target_component_sensitivity',
    'high_missingness_sensitivity',
    'pattern_variable_sensitivity',
}


def load_column_groups(manifest_path: str, scope: str = 'primary') -> dict[str, list[str]]:
    manifest = pd.read_csv(manifest_path)
    roles = PRIMARY_ROLES if scope == 'primary' else EXPANDED_ROLES
    if scope not in {'primary', 'expanded'}:
        raise ValueError("scope must be 'primary' or 'expanded'")
    selected = manifest[manifest['role'].isin(roles)]
    return {
        'numeric': selected.loc[selected['semantic_type'].eq('numeric'), 'column'].tolist(),
        'binary': selected.loc[selected['semantic_type'].eq('binary'), 'column'].tolist(),
        'categorical': selected.loc[selected['semantic_type'].eq('categorical'), 'column'].tolist(),
    }


def make_preprocessor(column_groups: dict[str, list[str]]) -> ColumnTransformer:
    numeric = Pipeline([
        ('imputer', SimpleImputer(strategy='median', add_indicator=True, keep_empty_features=True)),
        ('scaler', StandardScaler()),
    ])
    binary = Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent', add_indicator=True, keep_empty_features=True)),
    ])
    categorical = Pipeline([
        ('imputer', SimpleImputer(strategy='constant', fill_value='Unknown', keep_empty_features=True)),
        ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=True)),
    ])
    return ColumnTransformer([
        ('numeric', numeric, column_groups['numeric']),
        ('binary', binary, column_groups['binary']),
        ('categorical', categorical, column_groups['categorical']),
    ], remainder='drop', verbose_feature_names_out=True)
"""
factory_path = PREPROCESS_DIR / 'preprocessing_factory.py'
factory_path.write_text(factory_source, encoding='utf-8')
compile(factory_source, str(factory_path), 'exec')
print('Saved and syntax-checked:', factory_path)


## Automated post-cleaning audit

In [ ]:
# 13. Dataset-level and target-level audit checks
audit_rows = []

def record_check(check_id: str, description: str, passed: bool, details: str = ''):
    audit_rows.append({
        'check_id': check_id,
        'description': description,
        'status': 'PASS' if bool(passed) else 'FAIL',
        'details': details,
    })

record_check('A001', 'Source CSV checksum unchanged', source_hash_before == sha256_file(SOURCE_PATH), source_hash_before)
record_check('A002', 'Cleaned data retain all 820 subjects', len(clean) == 820, f'rows={len(clean)}')
record_check('A003', 'VisitID remains unique and complete', clean.VisitID.is_unique and clean.VisitID.notna().all())
record_check('A004', 'Exactly six agreed value corrections logged', len(correction_log) == 6, f'n={len(correction_log)}')
record_check('A005', 'MeanGripStrength was created', 'MeanGripStrength' in clean.columns)
record_check('A006', 'Only three retained outcomes configured', RETAINED_OUTCOMES == ['NegRFandACCP','NoHandJointInvolvement','AsymmetricArthritis'])
record_check('A007', 'OnlyLargeJoints excluded from retained outcomes', 'OnlyLargeJoints' not in RETAINED_OUTCOMES)
record_check('A008', 'No preprocessing object was globally fitted', all(
    not hasattr(make_preprocessor(preprocessing_specs[t]), 'transformers_') for t in RETAINED_OUTCOMES
))
record_check('A009', 'Source and clean outcomes are identical', all(raw[o].equals(clean[o]) for o in ALL_OUTCOMES))
record_check('A010', 'Decision YAML exists', DECISION_PATH.exists())
record_check('A011', 'Standalone preprocessing factory exists', (PREPROCESS_DIR / 'preprocessing_factory.py').exists())

for target in RETAINED_OUTCOMES:
    manifest = role_tables[target]
    primary_cols = set(manifest.loc[manifest.role.eq('primary_predictor'), 'column'])
    forbidden = (
        (set(ALL_OUTCOMES) - {target}) |
        constant_excluded | reproductive_excluded | high_missing_sensitivity |
        rare_excluded | overlap_alternative | pattern_sensitivity | grip_components |
        set(DECISIONS['target_specific_primary_exclusions'][target]) | {IDENTIFIER}
    )
    record_check(f'{target}_01', 'No forbidden columns in primary predictors', not bool(primary_cols & forbidden),
                 f'overlap={sorted(primary_cols & forbidden)}')
    record_check(f'{target}_02', 'Target is not a predictor', target not in primary_cols)
    record_check(f'{target}_03', 'TyG retained by explicit exception', 'TyG' in primary_cols)
    record_check(f'{target}_04', 'MeanGripStrength retained', 'MeanGripStrength' in primary_cols)
    record_check(f'{target}_05', 'Opium retained and IllicitDrug excluded',
                 'Opium' in primary_cols and 'IllicitDrug' not in primary_cols)
    record_check(f'{target}_06', 'Primary dataset exists',
                 (DERIVED_DIR / f'{target}__primary_eligible.csv').exists())
    record_check(f'{target}_07', 'Expanded dataset exists',
                 (DERIVED_DIR / f'{target}__sensitivity_expanded.csv').exists())
    record_check(f'{target}_08', 'Complete manifest exists',
                 (DERIVED_DIR / f'{target}__predictor_manifest.csv').exists())

audit = pd.DataFrame(audit_rows)
audit.to_csv(TABLE_DIR / 'stage2_validation_audit.csv', index=False, encoding='utf-8-sig')
display(audit)
if not audit.status.eq('PASS').all():
    failures = audit[audit.status.eq('FAIL')]
    raise AssertionError('Stage 2 audit failed:\n' + failures.to_string(index=False))
print(f'ALL {len(audit)} AUDIT CHECKS PASSED')


In [ ]:
# 14. Final artifact registry and run summary
artifact_rows = []
for category, directory in [
    ('config', CONFIG_DIR), ('cleaned_data', CLEAN_DIR), ('derived_data', DERIVED_DIR),
    ('preprocessing', PREPROCESS_DIR), ('validation_table', TABLE_DIR),
]:
    for path in sorted(directory.glob('*')):
        if path.is_file():
            artifact_rows.append({
                'category': category, 'filename': path.name,
                'path': str(path), 'size_bytes': path.stat().st_size,
                'sha256': sha256_file(path),
            })
artifact_registry = pd.DataFrame(artifact_rows)
artifact_registry.to_csv(TABLE_DIR / 'artifact_registry.csv', index=False, encoding='utf-8-sig')

after_clean_summary = pd.DataFrame([
    {'measure': 'source_rows', 'value': len(raw)},
    {'measure': 'source_columns', 'value': raw.shape[1]},
    {'measure': 'cleaned_rows', 'value': len(clean)},
    {'measure': 'cleaned_columns_including_MeanGripStrength', 'value': clean.shape[1]},
    {'measure': 'logged_corrections', 'value': len(correction_log)},
    {'measure': 'retained_outcomes', 'value': len(RETAINED_OUTCOMES)},
    {'measure': 'audit_checks_passed', 'value': int(audit.status.eq('PASS').sum())},
    {'measure': 'audit_checks_failed', 'value': int(audit.status.eq('FAIL').sum())},
])
after_clean_summary.to_csv(TABLE_DIR / 'stage2_run_summary.csv', index=False, encoding='utf-8-sig')
display(after_clean_summary)
display(artifact_registry[['category','filename','size_bytes']])

run_summary = {
    'decision_version': DECISIONS['version'],
    'source_sha256': source_hash_before,
    'source_rows': len(raw),
    'source_columns': raw.shape[1],
    'cleaned_rows': len(clean),
    'cleaned_columns': clean.shape[1],
    'retained_outcomes': RETAINED_OUTCOMES,
    'correction_count': len(correction_log),
    'all_audits_passed': bool(audit.status.eq('PASS').all()),
    'model_family_status': DECISIONS['model_family_status'],
}
with open(PROJECT_DIR / 'preprocessing_outputs' / 'stage2_run_summary.json', 'w', encoding='utf-8') as f:
    json.dump(run_summary, f, indent=2, ensure_ascii=False)


## Final cell — clean execution and HTML export

Save this notebook as:

```text
the authorized RA_PROJECT_DIR\02_data_cleaning_and_preprocessing.ipynb
```

Then run the final cell. It executes the saved notebook from a clean kernel and creates:

```text
the authorized RA_PROJECT_DIR\preprocessing_outputs\report\Stage2_cleaning_preprocessing_report.html
```


In [ ]:
# 15. FINAL CELL: execute all cells cleanly and export them to HTML
if os.environ.get('RA_STAGE2_EXPORT_MODE') == '1':
    print('Stage 2 HTML subprocess mode: recursive export intentionally skipped.')
else:
    candidates = [
        ENV_PROJECT / NOTEBOOK_NAME,
        Path.cwd() / NOTEBOOK_NAME,
        Path.cwd() / 'ra_analysis' / NOTEBOOK_NAME,
    ]
    notebook_path = next((p for p in candidates if p.exists()), None)
    if notebook_path is None:
        checked = '\n'.join(f'  - {p}' for p in candidates)
        raise FileNotFoundError(f'Save the notebook as {NOTEBOOK_NAME}. Checked:\n{checked}')

    env = os.environ.copy()
    env['RA_STAGE2_EXPORT_MODE'] = '1'
    command = [
        sys.executable, '-m', 'jupyter', 'nbconvert',
        '--to', 'html', '--execute',
        '--ExecutePreprocessor.timeout=1200',
        '--ExecutePreprocessor.kernel_name=python3',
        '--HTMLExporter.embed_images=True',
        '--output', 'Stage2_cleaning_preprocessing_report.html',
        '--output-dir', str(REPORT_DIR),
        str(notebook_path),
    ]
    print('Executing:', ' '.join(command))
    result = subprocess.run(command, env=env, text=True, capture_output=True)
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError(f'HTML export failed with exit code {result.returncode}')
    html_path = REPORT_DIR / 'Stage2_cleaning_preprocessing_report.html'
    if not html_path.exists():
        raise FileNotFoundError(f'Expected HTML was not created: {html_path}')
    print('HTML report created:', html_path.resolve())
